In [1]:
import pandas as pd 
import numpy as np 

In [2]:
metadata = pd.read_csv('../../data/metadata_demographic.csv')
metadata = metadata.drop_duplicates(subset='Participant_ID')
metadata = metadata.dropna(subset=['pd'])
metadata

,Protocol,Participant_ID,Task,gender,age,race,pd
0,SuperPD,NIHFT628PHTAY,ahhhh,female,70.0,White,no
1,SuperPD,NIHNT179KNNF4,ahhhh,female,70.0,White,yes
2,SuperPD,NIHYM875FLXFF,ahhhh,female,73.0,White,no
3,SuperPD,NIHBV117HUCTC,ahhhh,female,60.0,White,no
4,SuperPD,NIHZY217YWJA8,ahhhh,female,69.0,White,yes
...,...,...,...,...,...,...,...
1996,ValorPD,MOyJjyLX9hPvP3FSlLNYaG28xA23,NaN,M,78.0,White,yes
1997,ValorPD,zn6iI7uiq0U0xWTG5fuwR7yX9IH3,NaN,F,62.0,White,no
1998,ValorPD,XFJkSNMEpNgUnWg5Ouc6AWMOnQ82,NaN,F,62.0,White,yes
1999,ValorPD,j4GZI8ZFugZHRPxCKLBHC3DFwZE2,NaN,M,70.0,White,yes


In [3]:
fusion_ids = pd.read_csv('../../data/fusion_ids_full_cohort.csv')
fusion_ids = fusion_ids.drop_duplicates(subset='id')
fusion_ids.shape

(681, 2)

In [4]:
metadata = metadata[metadata['Participant_ID'].isin(fusion_ids['id'])]
metadata.shape

(681, 7)

In [5]:
metadata['pd'].value_counts(dropna=False)

pd
no          622
Unlikely     57
yes           2
Name: count, dtype: int64

In [6]:
# Mapping the 'pd' column to binary values (0 and 1)
# Assuming 'no', 'Control', and 'Unlikely' are mapped to 0 (non-PD)
# and the rest are mapped to 1 (PD)

pd_map = {
    'no': 0,
    'Control': 0,
    'Unlikely': 0,
    'yes': 1,
    'Possible': 1,
    'PD': 1,
    'Probable': 1
}

metadata['Diagnosis'] = metadata['pd'].map(pd_map)
metadata['Diagnosis'].value_counts(dropna=False)

/tmp/ipykernel_2543660/3990877965.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  metadata['Diagnosis'] = metadata['pd'].map(pd_map)


Diagnosis
0    679
1      2
Name: count, dtype: int64

In [7]:
def create_distribution_table_with_order(df, diagnosis_col, target_col, first_row_text, value_order=None):
    """
    Create a table showing counts and percentages for a specified column split by a diagnosis column.
    
    Parameters:
        df (pd.DataFrame): The input dataset.
        diagnosis_col (str): The column that indicates diagnosis (e.g., "Diagnosis").
        target_col (str): The column for which distribution is calculated (e.g., "Sex").
        first_row_text (str): The first row text for the table header (e.g., "Sex, n (%)").
        value_order (list): The preferred order of column values (e.g., ['Male', 'Female', ...]).
        
    Returns:
        pd.DataFrame: The distribution table.
    """
    # Calculate counts within each group (With PD and Without PD)
    with_pd_total = df[df[diagnosis_col] == 1].shape[0]
    without_pd_total = df[df[diagnosis_col] != 1].shape[0]

    # Counts for each target value in the groups
    with_pd_counts = df[df[diagnosis_col] == 1][target_col].value_counts()
    without_pd_counts = df[df[diagnosis_col] != 1][target_col].value_counts()
    total_counts = df[target_col].value_counts()

    # Use value_order or infer from the data
    if value_order is None:
        value_order = total_counts.index.tolist()

    # Create the rows for each unique value in the preferred order
    rows = []
    for value in value_order:
        # Get counts for each group
        with_pd_count = with_pd_counts.get(value, 0)
        without_pd_count = without_pd_counts.get(value, 0)
        total_count = total_counts.get(value, 0)

        # Calculate percentages within each group
        with_pd_pct = (with_pd_count / with_pd_total) * 100 if with_pd_total > 0 else 0
        without_pd_pct = (without_pd_count / without_pd_total) * 100 if without_pd_total > 0 else 0
        total_pct = (total_count / df.shape[0]) * 100

        # Add row to the table
        rows.append([
            first_row_text,
            value,
            f"{with_pd_count} ({with_pd_pct:.1f}%)",
            f"{without_pd_count} ({without_pd_pct:.1f}%)",
            f"{total_count} ({total_pct:.1f}%)"
        ])

    # Create the table DataFrame
    table = pd.DataFrame(rows, columns=["Demographic Property", "Attribute", "LRRK2 Carrier", "Control", "Total"])

    # # Add the first row text
    # first_row = pd.DataFrame([{
    #     "Demographic Property": first_row_text,
    #     "Attribute": "",
    #     "With PD": "",
    #     "Without PD": "",
    #     "Total": ""
    # }])

    # # Concatenate the header and the data rows
    # table = pd.concat([first_row, table], ignore_index=True)

    for index in range(1, len(table)):
        table.loc[index, 'Demographic Property'] = ''
    
    return table


In [8]:
metadata['Protocol'].value_counts(dropna=False)

Protocol
ParkTest           400
ValorPD            122
ClusterPD           57
ValidationStudy     47
SuperPD_old         23
InMotion            20
SuperPD             12
Name: count, dtype: int64

In [9]:
metadata['carrier'] = 0
# take all Protocol = ValorPD, and set carrier = 1
metadata.loc[metadata['Protocol'] == 'ValorPD', 'carrier'] = 1

/tmp/ipykernel_2543660/2910501634.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  metadata['carrier'] = 0


In [10]:
metadata['carrier'].value_counts(dropna=False)

carrier
0    559
1    122
Name: count, dtype: int64

In [11]:
# Calculate counts and percentages for 'With PD' and 'Without PD'
with_pd_count = (metadata['carrier'] == 1).sum()  # Assuming '1' indicates 'With PD'
without_pd_count = (metadata['carrier'] != 1).sum()
total_count = with_pd_count + without_pd_count

with_pd_percentage = (with_pd_count / total_count) * 100
without_pd_percentage = (without_pd_count / total_count) * 100

# Create the final table structure
table_dict = {
    "Demographic Property": "",
    "Attribute": ["Number of Participants, n (%)"],
    "LRRK2 Carrier": [f"{with_pd_count} ({with_pd_percentage:.1f}%)"],
    "Control": [f"{without_pd_count} ({without_pd_percentage:.1f}%)"],
    "Total": [f"{total_count} (100%)"]
}

# Convert to a DataFrame for display
pd_count_table = pd.DataFrame(table_dict)

pd_count_table

,Demographic Property,Attribute,LRRK2 Carrier,Control,Total
0,,"Number of Participants, n (%)",122 (17.9%),559 (82.1%),681 (100%)


In [12]:
metadata['gender'].value_counts()

gender
female    282
male      230
F          71
M          51
Female     29
Male       18
Name: count, dtype: int64

In [13]:
gender_map = {
    "female": "Female",
    "Female": "Female",
    'F': 'Female',
    "male": "Male",
    "Male": "Male",
    'M': 'Male',
    "Prefer not to respond": "Unknown",
    "Nonbinary": "Non-Binary"
}
metadata['gender_normalized'] = metadata['gender'].map(gender_map)
metadata['gender_normalized'].value_counts(dropna=False)

/tmp/ipykernel_2543660/2488056917.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  metadata['gender_normalized'] = metadata['gender'].map(gender_map)


gender_normalized
Female    382
Male      299
Name: count, dtype: int64

In [14]:
# Define the preferred order
preferred_order = ['Female', 'Male',]

# Call the function with the preferred order
sex_table_with_order = create_distribution_table_with_order(
    df=metadata,
    diagnosis_col="carrier",
    target_col="gender_normalized",
    first_row_text="Sex",
    value_order=preferred_order
)


sex_table_with_order

,Demographic Property,Attribute,LRRK2 Carrier,Control,Total
0,Sex,Female,71 (58.2%),311 (55.6%),382 (56.1%)
1,,Male,51 (41.8%),248 (44.4%),299 (43.9%)


In [15]:
metadata['age'].value_counts(dropna=False)

age
NaN     57
60.0    47
66.0    42
59.0    37
57.0    36
        ..
16.0     1
54.0     1
26.0     1
78.0     1
44.0     1
Name: count, Length: 66, dtype: int64

In [16]:
metadata['age'] = metadata['age'].apply(lambda x: np.nan if x < 15 or x > 100 else x)
# Display the value counts of the 'age' column sorted by ascending order of the age values
metadata['age'].value_counts(dropna=False).sort_index()


/tmp/ipykernel_2543660/2598866290.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  metadata['age'] = metadata['age'].apply(lambda x: np.nan if x < 15 or x > 100 else x)


age
16.0     1
17.0     2
18.0     1
19.0     1
20.0     1
        ..
79.0     1
80.0     1
85.0     1
87.0     2
NaN     57
Name: count, Length: 66, dtype: int64

In [17]:
# Initialize 'age_normalized' column with NaN
metadata['age_normalized'] = np.nan

# Ensure 'age' is numeric where possible
def safe_numeric(x):
    try:
        return float(x)
    except (ValueError, TypeError):
        return np.nan

metadata['age_numeric'] = metadata['age'].apply(safe_numeric)

# Define conditions for numeric age ranges
conditions = [
    metadata['age_numeric'] < 20,
    (metadata['age_numeric'] >= 20) & (metadata['age_numeric'] <= 39),
    (metadata['age_numeric'] >= 40) & (metadata['age_numeric'] <= 59),
    (metadata['age_numeric'] >= 60) & (metadata['age_numeric'] <= 79),
    metadata['age_numeric'] >= 80
]

# Define labels for the conditions
age_labels = [
    '< 20',
    '20 - 39',
    '40 - 59',
    '60 - 79',
    '>= 80'
]

# Apply conditions to normalize 'age'
metadata['age_normalized'] = np.select(
    conditions,
    age_labels,
    default='Not Mentioned'
)


/tmp/ipykernel_2543660/3161752875.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  metadata['age_normalized'] = np.nan
/tmp/ipykernel_2543660/3161752875.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  metadata['age_numeric'] = metadata['age'].apply(safe_numeric)
/tmp/ipykernel_2543660/3161752875.py:32: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://panda

In [18]:
import numpy as np

def process_age(age):
    try:
        # If the age is already numeric, return it
        if isinstance(age, (int, float)):
            return float(age)
        # If the age is a range like "50-60", calculate the mean
        if isinstance(age, str) and '-' in age:
            start, end = map(float, age.split('-'))
            return (start + end) / 2
    except:
        pass
    # Return NaN for invalid entries
    return np.nan

# Apply the processing function to handle all cases
metadata['age_processed'] = metadata['age'].apply(process_age)

# Calculate the mean and range (ignoring NaN values)
mean_age = metadata['age_processed'].mean()
min_age = metadata['age_processed'].min()
max_age = metadata['age_processed'].max()

# Print the results
print(f"Mean age: {mean_age:.2f}")
print(f"Age range: {min_age:.2f} - {max_age:.2f}")


Mean age: 57.94
Age range: 16.00 - 87.00


/tmp/ipykernel_2543660/563915863.py:18: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  metadata['age_processed'] = metadata['age'].apply(process_age)


In [19]:
metadata['age_normalized'].value_counts(dropna=False)

age_normalized
60 - 79          358
40 - 59          175
20 - 39           82
Not Mentioned     57
< 20               5
>= 80              4
Name: count, dtype: int64

In [20]:
# Define the preferred order
preferred_order = [
    '< 20',
    '20 - 39',
    '40 - 59',
    '60 - 79',
    '>= 80',
    'Not Mentioned'
]



# Call the function with the preferred order
age_table_with_order = create_distribution_table_with_order(
    df=metadata,
    diagnosis_col="carrier",
    target_col="age_normalized",
    first_row_text=f"Age in years (range: {min_age:.1f} - {max_age:.1f}, mean: {mean_age:.1f}), n (%)",
    value_order=preferred_order
)


age_table_with_order

,Demographic Property,Attribute,LRRK2 Carrier,Control,Total
0,"Age in years (range: 16.0 - 87.0, mean: 57.9),...",< 20,0 (0.0%),5 (0.9%),5 (0.7%)
1,,20 - 39,35 (28.7%),47 (8.4%),82 (12.0%)
2,,40 - 59,38 (31.1%),137 (24.5%),175 (25.7%)
3,,60 - 79,49 (40.2%),309 (55.3%),358 (52.6%)
4,,>= 80,0 (0.0%),4 (0.7%),4 (0.6%)
5,,Not Mentioned,0 (0.0%),57 (10.2%),57 (8.4%)


In [21]:
metadata['race'].value_counts(dropna=False)

race
white,                           296
White                            207
['White']                         41
white,race                        34
black,                            24
asian,race                        21
NaN                               20
Prefer Not to Answer               6
asian,                             5
on,                                5
asian,white,                       4
other,race                         3
black,race                         3
white                              2
white,black,                       2
['Asian']                          2
['White', 'Asian']                 1
['Asian', 'White']                 1
black                              1
Black or African American          1
['Black or African American']      1
['Prefer not to respond']          1
Name: count, dtype: int64

In [22]:
def map_race_simplified(value):
    if isinstance(value, str):
        value = value.lower()  # Make case-insensitive
        if 'asian' in value:
            return "Asian"
        elif 'nativeamerican' in value or 'american indian' in value:
            return "American Indian or Alaska Native"
        elif 'black' in value:
            return "Black or African American"
        elif 'nativepacific' in value or 'hawaiian' in value:
            return "Native Hawaiian or Other Pacific Islander"
        elif 'white' in value:
            return "White"
        elif 'other' in value or 'on' in value:
            return "Others"
        elif 'prefer not to' in value or '[]' in value:
            return "Not Mentioned"
    return "Not Mentioned"  # Default to Unknown if value is not a string or doesn't match

# Apply the function to the 'race' column
metadata['race_normalized'] = metadata['race'].apply(map_race_simplified)

# Display the cleaned race column counts
print(metadata['race_normalized'].value_counts(dropna=False))

race_normalized
White                        580
Asian                         34
Black or African American     32
Not Mentioned                 26
Others                         9
Name: count, dtype: int64


/tmp/ipykernel_2543660/2427306195.py:21: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  metadata['race_normalized'] = metadata['race'].apply(map_race_simplified)


In [23]:
# Define the preferred order
preferred_order = [
    "White",
    "Asian",
    "Black or African American",
    "American Indian or Alaska Native",
    "Others",
    "Not Mentioned"
]

# Call the function with the preferred order
race_table_with_order = create_distribution_table_with_order(
    df=metadata,
    diagnosis_col="carrier",
    target_col="race_normalized",
    first_row_text="Race, n (%)",
    value_order=preferred_order
)


race_table_with_order

,Demographic Property,Attribute,LRRK2 Carrier,Control,Total
0,"Race, n (%)",White,116 (95.1%),464 (83.0%),580 (85.2%)
1,,Asian,0 (0.0%),34 (6.1%),34 (5.0%)
2,,Black or African American,0 (0.0%),32 (5.7%),32 (4.7%)
3,,American Indian or Alaska Native,0 (0.0%),0 (0.0%),0 (0.0%)
4,,Others,0 (0.0%),9 (1.6%),9 (1.3%)
5,,Not Mentioned,6 (4.9%),20 (3.6%),26 (3.8%)


In [24]:
metadata['Protocol'].value_counts(dropna=False)

Protocol
ParkTest           400
ValorPD            122
ClusterPD           57
ValidationStudy     47
SuperPD_old         23
InMotion            20
SuperPD             12
Name: count, dtype: int64

In [25]:
env_map = {
    'ParkTest':'Home-Global',
    'ValorPD': 'Clinic',
    'InMotion': 'PD Care Facility',
    'ValidationStudy': 'Clinic',
    'SuperPD': 'Clinic',
    'ClusterPD': 'Clinic',
    'SuperPD_old': 'Clinic',
    'SuperPD': 'Clinic',
    'RoutePD': 'PD Care Facility'
    
}

metadata['env'] = metadata['Protocol'].map(env_map)
metadata['env'].value_counts(dropna=False)

/tmp/ipykernel_2543660/1323468674.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  metadata['env'] = metadata['Protocol'].map(env_map)


env
Home-Global         400
Clinic              261
PD Care Facility     20
Name: count, dtype: int64

In [26]:
# Define the preferred order
preferred_order = [
    "Home-Global",
    "PD Care Facility",
    "Clinic",
]

# Call the function with the preferred order
env_table_with_order = create_distribution_table_with_order(
    df=metadata,
    diagnosis_col="carrier",
    target_col="env",
    first_row_text="Recording Environment, n (%)",
    value_order=preferred_order
)
env_table_with_order

,Demographic Property,Attribute,LRRK2 Carrier,Control,Total
0,"Recording Environment, n (%)",Home-Global,0 (0.0%),400 (71.6%),400 (58.7%)
1,,PD Care Facility,0 (0.0%),20 (3.6%),20 (2.9%)
2,,Clinic,122 (100.0%),139 (24.9%),261 (38.3%)


In [27]:
total_table = pd.concat([
    pd_count_table, 
    sex_table_with_order, 
    age_table_with_order, 
    race_table_with_order, 
    env_table_with_order
], ignore_index=True)
total_table

,Demographic Property,Attribute,LRRK2 Carrier,Control,Total
0,,"Number of Participants, n (%)",122 (17.9%),559 (82.1%),681 (100%)
1,Sex,Female,71 (58.2%),311 (55.6%),382 (56.1%)
2,,Male,51 (41.8%),248 (44.4%),299 (43.9%)
3,"Age in years (range: 16.0 - 87.0, mean: 57.9),...",< 20,0 (0.0%),5 (0.9%),5 (0.7%)
4,,20 - 39,35 (28.7%),47 (8.4%),82 (12.0%)
5,,40 - 59,38 (31.1%),137 (24.5%),175 (25.7%)
6,,60 - 79,49 (40.2%),309 (55.3%),358 (52.6%)
7,,>= 80,0 (0.0%),4 (0.7%),4 (0.6%)
8,,Not Mentioned,0 (0.0%),57 (10.2%),57 (8.4%)
9,"Race, n (%)",White,116 (95.1%),464 (83.0%),580 (85.2%)


In [28]:
total_table.to_csv('../../data/demographic_table_full_cohort.csv', index=False)  

In [29]:
# save the metadata
metadata.to_csv('../../data/metadata_full_cohort.csv', index=False)

In [30]:
# calculate all the participants 
ufnet_participants = metadata
ufnet_participants

,Protocol,Participant_ID,Task,gender,age,race,pd,Diagnosis,carrier,gender_normalized,age_normalized,age_numeric,age_processed,race_normalized,env
0,SuperPD,NIHFT628PHTAY,ahhhh,female,70.0,White,no,0,0,Female,60 - 79,70.0,70.0,White,Clinic
2,SuperPD,NIHYM875FLXFF,ahhhh,female,73.0,White,no,0,0,Female,60 - 79,73.0,73.0,White,Clinic
3,SuperPD,NIHBV117HUCTC,ahhhh,female,60.0,White,no,0,0,Female,60 - 79,60.0,60.0,White,Clinic
5,SuperPD,NIHTN717JDEYY,eye_gaze,female,50.0,White,no,0,0,Female,40 - 59,50.0,50.0,White,Clinic
6,SuperPD,NIHHY970PYCFA,quick_brown_fox,female,69.0,White,no,0,0,Female,60 - 79,69.0,69.0,White,Clinic
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1990,ValorPD,UWVRZ5jOPXTXdOaFsF5FJdzzgwL2,NaN,F,61.0,White,no,0,1,Female,60 - 79,61.0,61.0,White,Clinic
1991,ValorPD,14GYE2DIFeRiie4h48KtjxOaijm1,NaN,F,29.0,White,no,0,1,Female,20 - 39,29.0,29.0,White,Clinic
1992,ValorPD,oxQnpE2jM2PG2MNDQZMmzARYRlj1,NaN,F,36.0,White,no,0,1,Female,20 - 39,36.0,36.0,White,Clinic
1993,ValorPD,rJG6ZYa8OASB3bxczHFNiQulmyi1,NaN,F,34.0,White,no,0,1,Female,20 - 39,34.0,34.0,White,Clinic


In [31]:
df_pd_weigh_in = pd.read_csv("../../data/clinical_matched_finger_tapping_smile_qbf_with_features.csv")
df_pd_weigh_in = df_pd_weigh_in.drop_duplicates(subset='participant_id')
df_pd_weigh_in.head()

,participant_id,visit_date_park,visit_date_clinical,protocol_park,sex,age,dob,race,ethnicity,diagnosis,...,cepj11,cepj12,Hnorm,alpha,ppe,relbandpower0,relbandpower1,relbandpower2,relbandpower3,f0std
0,0etnk2RTVUf7PwaB0y05pV5TW3k1,9/26/22,9/26/22,valor_pd,Male,60.060000,9/3/62,Asian,Not Hispanic/Latino,1,...,101.271344,95.041335,0.892602,1.753825,0.605036,0.0,-0.980468,-1.535022,-2.255130,102.409434
2,0jZnljtTHgh8GHCJ7GCKa0MNF932,3/21/22,3/21/22,valor_pd,Male,45.380000,11/5/76,White,Not Hispanic/Latino,0,...,86.554700,95.564922,0.830559,1.484562,0.687275,0.0,-0.864954,-1.222613,-1.334499,143.664579
5,0lG6wpFlYPSyEIFPxpINpGrwLBc2,11/2/23,11/2/23,route_pd,Female,79.300735,7/15/44,Native Hawaiian or Other Pacific Islander,Not Hispanic/Latino,1,...,118.281884,123.711903,0.872175,1.476042,0.838535,0.0,-0.558553,-0.721594,-1.584893,100.676062
7,0Y4nmh735kMWaP6srYZpR2SN8ns1,11/18/21,11/18/21,valor_pd,Female,34.330000,7/18/87,White,Not Hispanic/Latino,0,...,104.564659,108.843143,0.872309,1.369875,0.807490,0.0,-0.473624,-0.872065,-1.929557,109.703043
10,13dnQ0NpHnOdwr1PO6278fwLLYG3,2/17/22,2/17/22,valor_pd,Male,80.830000,4/17/41,White,Not Hispanic/Latino,1,...,85.295530,78.272439,0.856144,1.920971,0.707185,0.0,-0.814351,-1.648895,-2.166235,111.052831


In [32]:
ufnet_participants_ids = metadata['Participant_ID'].unique()
df_pd_weigh_in_ids = df_pd_weigh_in['participant_id'].unique()

len(ufnet_participants_ids), len(df_pd_weigh_in_ids)

(681, 304)

In [33]:
len(set(ufnet_participants_ids).intersection(set(df_pd_weigh_in_ids)))

156

In [34]:
len(set(ufnet_participants_ids).union(set(df_pd_weigh_in_ids)))

829

In [35]:
# take id and protocol from both
ufnet_participants_info = metadata[metadata['Participant_ID'].isin(ufnet_participants_ids)][['Participant_ID', 'Protocol', 'Diagnosis']]
df_pd_weigh_in_info = df_pd_weigh_in[df_pd_weigh_in['participant_id'].isin(df_pd_weigh_in_ids)][['participant_id', 'protocol_park', 'diagnosis']]

# rename columns to make them consistent
df_pd_weigh_in_info.columns = ['Participant_ID', 'Protocol', 'Diagnosis']

# add location column
ufnet_participants_info['Location'] = 'UFNet'
df_pd_weigh_in_info['Location'] = 'PD Weigh-In'


In [36]:
ufnet_participants_info['Protocol'].value_counts(dropna=False), df_pd_weigh_in_info['Protocol'].value_counts(dropna=False)

(Protocol
 ParkTest           400
 ValorPD            122
 ClusterPD           57
 ValidationStudy     47
 SuperPD_old         23
 InMotion            20
 SuperPD             12
 Name: count, dtype: int64,
 Protocol
 valor_pd    158
 super_pd     99
 route_pd     47
 Name: count, dtype: int64)

In [37]:
# remove _
df_pd_weigh_in_info['Protocol'] = df_pd_weigh_in_info['Protocol'].str.replace('_', '')

# capitalize
df_pd_weigh_in_info['Protocol'] = df_pd_weigh_in_info['Protocol'].str.title()

# make the term pd allcaps
df_pd_weigh_in_info['Protocol'] = df_pd_weigh_in_info['Protocol'].str.replace('pd', 'PD', case=False)

df_pd_weigh_in_info['Protocol'].value_counts(dropna=False)


Protocol
ValorPD    158
SuperPD     99
RoutePD     47
Name: count, dtype: int64

In [38]:
full_data = pd.concat([ufnet_participants_info, df_pd_weigh_in_info], ignore_index=True)
full_data.to_csv('../../data/demographic_full_participant_list.csv', index=False)

In [39]:
full_data['Participant_ID'].nunique()

829

In [40]:
df_repeated_ids = full_data[full_data.duplicated(['Participant_ID'], keep=False)]
df_repeated_ids.drop_duplicates(subset=['Participant_ID'], inplace=True)
df_repeated_ids['Protocol'].value_counts()

/tmp/ipykernel_2543660/720013172.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_repeated_ids.drop_duplicates(subset=['Participant_ID'], inplace=True)


Protocol
ValorPD        122
SuperPD_old     22
SuperPD         12
Name: count, dtype: int64